# Rappi Operations: Exploratory Data Analysis

This notebook performs an initial exploration of the operational metrics provided by Rappi and answers the 6 key business scenarios required for the technical interview.

In [1]:
import sys
import os
sys.path.append(os.path.abspath('../src'))

import pandas as pd
import plotly.express as px
import plotly.io as pio
from data_loader import DataLoader

# Set template for plotly
pio.templates.default = "plotly_white"

# Load data
loader = DataLoader("../data/dummy_data.xlsx")
df = loader.get_data()

print(f"Loaded {len(df)} rows of data.")
df.head()

Loaded 124335 rows of data.


,COUNTRY,CITY,ZONE,ZONE_TYPE,ZONE_PRIORITIZATION,METRIC,WEEK,VALUE,WEEK_NUM
0,EC,Quito,San Martin de Porras,Non Wealthy,Not Prioritized,Retail SST > SS CVR,L8W,0.602339,8
1,BR,Bauru,FS Parque Jardim Europa/Vila Riachuelo - BAU,Non Wealthy,Not Prioritized,Restaurants SST > SS CVR,L8W,0.857544,8
2,MX,Mexicali,MXL_Universidad,Wealthy,Prioritized,Retail SST > SS CVR,L8W,0.900823,8
3,CL,Santiago De Chile,Colina,Non Wealthy,Not Prioritized,Retail SST > SS CVR,L8W,0.798946,8
4,MX,Guadalajara,Valle Real,Wealthy,High Priority,Gross Profit UE,L8W,3.914477,8


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 124335 entries, 0 to 124334
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   COUNTRY              124335 non-null  str    
 1   CITY                 124335 non-null  str    
 2   ZONE                 124335 non-null  str    
 3   ZONE_TYPE            121959 non-null  str    
 4   ZONE_PRIORITIZATION  121959 non-null  str    
 5   METRIC               124335 non-null  str    
 6   WEEK                 124335 non-null  str    
 7   VALUE                121526 non-null  float64
 8   WEEK_NUM             124335 non-null  int64  
dtypes: float64(1), int64(1), str(7)
memory usage: 17.2 MB


In [5]:
for col in df.columns:
    print(f"{df[col].value_counts()}")

COUNTRY
MX    39339
BR    26928
CO    16560
AR    12465
PE    10755
EC     8838
CL     7110
CR     1449
UY      891
Name: count, dtype: int64
CITY
Buenos Aires        5148
Lima                4995
Rio De Janeiro      4257
Grande São Paulo    3789
Quito               3528
                    ... 
Itu                    9
Nacajuca               9
Atibaia                9
Betim                  9
Tarapoto               9
Name: count, Length: 342, dtype: int64
ZONE
Centro                           936
Norte                            270
Independencia                    270
Colina                           261
Miraflores                       261
                                ... 
PR - FOZ - Jardim Itamarati        9
MP VILA LIMEIRANEA LIM             9
MP Ponta do Farol/ Calhau SLZ      9
Maringá - VILLA EMILIA             9
Barrio Comercio                    9
Name: count, Length: 1190, dtype: int64
ZONE_TYPE
Non Wealthy    95670
Wealthy        26289
Name: count, dtype: int64
ZONE_PRIO

In [7]:
df[df.isna()].sample(10)

,COUNTRY,CITY,ZONE,ZONE_TYPE,ZONE_PRIORITIZATION,METRIC,WEEK,VALUE,WEEK_NUM
75515,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
68664,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
66946,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
55687,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27962,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
72758,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35789,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13796,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9045,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23453,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1. Top 5 Zones by Lead Penetration (Current Week)
**Question:** Which are the 5 zones with more % Lead Penetration this week?

In [2]:
lead_pen = df[(df['METRIC'] == 'Lead Penetration') & (df['WEEK_NUM'] == 0)]
top_5_lead_pen = lead_pen.sort_values(by='VALUE', ascending=False).head(5)

fig1 = px.bar(top_5_lead_pen, x='ZONE', y='VALUE', color='CITY', 
             title="Top 5 Zones by Lead Penetration (L0W)",
             labels={'VALUE': 'Lead Penetration %', 'ZONE': 'Zone'})
fig1.show()
top_5_lead_pen[['COUNTRY', 'CITY', 'ZONE', 'VALUE']]

,COUNTRY,CITY,ZONE,VALUE
111614,EC,Quito,Sur de Quito,393.9
112140,EC,Quito,Valle de los Chillos,158.6
103981,EC,Quito,Antiguo Aeropuerto,141.3
111892,EC,Quito,Vicentina- Floresta,87.8
112228,EC,Guayaquil,Via la Costa Oeste,57.2


## 2. Perfect Order Comparison: Wealthy vs Non Wealthy (Mexico)
**Question:** Compare the Perfect Order metric between Wealthy and Non Wealthy zones in Mexico.

In [8]:
mx_perfect = df[(df['METRIC'] == 'Perfect Orders') & (df['COUNTRY'] == 'MX')]
mx_comparison = mx_perfect.groupby('ZONE_TYPE')['VALUE'].mean().reset_index()

fig2 = px.bar(mx_comparison, x='ZONE_TYPE', y='VALUE', color='ZONE_TYPE',
             title="Average Perfect Orders in Mexico by Zone Type",
             labels={'VALUE': 'Avg Perfect Order %', 'ZONE_TYPE': 'Zone Wealth Category'})
fig2.show()
mx_comparison

,ZONE_TYPE,VALUE
0,Non Wealthy,0.862805
1,Wealthy,0.899902


## 3. Temporal Evolution: Gross Profit UE in Chapinero
**Question:** Show the evolution of Gross Profit UE in Chapinero for the last 8 weeks.

In [9]:
chapinero_profit = df[(df['METRIC'] == 'Gross Profit UE') & (df['ZONE'] == 'Chapinero')]
# Sort by week number descending (from 8 to 0)
chapinero_profit = chapinero_profit.sort_values(by='WEEK_NUM', ascending=False)

fig3 = px.line(chapinero_profit, x='WEEK', y='VALUE', markers=True,
              title="Gross Profit UE Evolution in Chapinero",
              labels={'VALUE': 'Gross Profit UE', 'WEEK': 'Weeks Ago (L8W to L0W)'})
fig3.show()

## 4. Aggregations: Avg Lead Penetration per Country
**Question:** What is the average Lead Penetration per country?

In [10]:
country_lead = df[df['METRIC'] == 'Lead Penetration'].groupby('COUNTRY')['VALUE'].mean().reset_index()
country_lead = country_lead.sort_values(by='VALUE', ascending=False)

fig4 = px.bar(country_lead, x='COUNTRY', y='VALUE', 
             title="Average Lead Penetration per Country",
             labels={'VALUE': 'Avg Lead Penetration %'})
fig4.show()

## 5. Multivariate Analysis: High Lead Pen vs Low Perfect Order
**Question:** Which zones have higher Lead Penetration but low Perfect Order?

In [11]:
# Filter and pivot current week metrics
pivot_df = df[df['WEEK_NUM'] == 0].pivot_table(
    index=['COUNTRY', 'CITY', 'ZONE'], 
    columns='METRIC', 
    values='VALUE'
).reset_index()

# Define thresholds (e.g. median)
lp_median = pivot_df['Lead Penetration'].median()
po_median = pivot_df['Perfect Orders'].median()

high_lp_low_po = pivot_df[
    (pivot_df['Lead Penetration'] > lp_median) & 
    (pivot_df['Perfect Orders'] < po_median)
]

fig5 = px.scatter(pivot_df, x='Lead Penetration', y='Perfect Orders', 
                 hover_name='ZONE', color='COUNTRY', 
                 title="Lead Penetration vs Perfect Orders (L0W)")
fig5.add_hline(y=po_median, line_dash="dot", annotation_text="Median PO")
fig5.add_vline(x=lp_median, line_dash="dot", annotation_text="Median LP")
fig5.show()

print(f"Found {len(high_lp_low_po)} zones with High LP and Low PO.")
high_lp_low_po[['COUNTRY', 'CITY', 'ZONE', 'Lead Penetration', 'Perfect Orders']].head()

Found 172 zones with High LP and Low PO.


METRIC,COUNTRY,CITY,ZONE,Lead Penetration,Perfect Orders
2,AR,Bariloche,BARILOCHE_CENTRO,0.248512,0.866870
3,AR,Buenos Aires,Avellaneda - Sarandi,0.287480,0.861389
4,AR,Buenos Aires,BERAZATEGUI,0.170810,0.843646
6,AR,Buenos Aires,Boulogne,0.620694,0.866160
10,AR,Buenos Aires,Caseros - Bonich,0.189821,0.874488


## 6. Inference: Rapid Growth in Orders
**Question:** Which zones are the most rapidly growing in orders and what could explain this growth?

In [12]:
orders_df = df[df['METRIC'] == 'Orders'].pivot_table(
    index=['COUNTRY', 'CITY', 'ZONE'], 
    columns='WEEK', 
    values='VALUE'
).reset_index()

# Calculate growth from 8 weeks ago to current week
orders_df['Growth'] = (orders_df['L0W'] - orders_df['L8W']) / orders_df['L8W']

top_growth = orders_df.sort_values(by='Growth', ascending=False).head(10)

# To explain growth, let's look at other metrics for these top zones
top_zones = top_growth['ZONE'].tolist()
explanation_metrics = pivot_df[pivot_df['ZONE'].isin(top_zones)]

fig6 = px.bar(top_growth, x='ZONE', y='Growth', title="Top 10 Zones by Order Growth (L8W to L0W)")
fig6.show()

print("Top Growth Zones Metrics Summary (Current Week):")
explanation_metrics.describe().loc[['mean', '50%']]

Top Growth Zones Metrics Summary (Current Week):


METRIC,% PRO Users Who Breakeven,% Restaurants Sessions With Optimal Assortment,Gross Profit UE,Lead Penetration,MLTV Top Verticals Adoption,Non-Pro PTC > OP,Orders,Perfect Orders,Pro Adoption (Last Week Status),Restaurants Markdowns / GMV,Restaurants SS > ATC CVR,Restaurants SST > SS CVR,Retail SST > SS CVR,Turbo Adoption
mean,0.188086,0.067398,-0.482944,0.059819,0.051044,0.685737,559.722222,0.834148,0.200194,0.190800,0.410361,0.800470,0.682490,NaN
50%,0.198718,0.000000,-0.862311,0.047827,0.059371,0.698289,17.000000,0.871429,0.213838,0.166971,0.420176,0.854864,0.849145,NaN
